# Sparsity-Aware Reactive Obstacle Avoidance Network Training

This notebook implements a **sparsity-aware PointNet-based reactive correction network** that learns both **when** and **how** to apply QP filter corrections for obstacle avoidance.

## 🎯 Key Innovation: Sparsity Learning

Unlike the original approach that always applies corrections, this version learns **sparse activations** matching expert QP filter behavior:

- **QP Inactive** (`||Δq|| ≤ 0.05`): Model outputs near-zero corrections (weight = 0.2)
- **QP Active** (`||Δq|| > 0.05`): Model learns accurate corrections (weight = 1.0)

## Architecture Overview:
- **Input**: Point cloud P ∈ R^{1500×3} (robot frame) + joint state q ∈ R^6
- **Encoder**: Per-point MLP (3 → 64 → 128) + Max-pool → global feature f_pc ∈ R^128
- **Fusion**: Concatenate [f_pc, q] → MLP 134 → 128 → 64 → 6
- **Output**: Residual Δq = action_raw - action_processed
- **Training**: **Weighted MSE loss** with sparsity-aware sample weighting

## 🚀 Expected Improvements:
1. **Sparse Activation**: Only corrects near obstacles (like expert QP filter)
2. **Better Correlation**: High correlation with expert correction timing
3. **Stable Control**: No over-correction in free space
4. **Improved Success Rate**: Better performance across all obstacle heights

The goal is to learn **both the binary decision** (should I intervene?) **and the correction magnitude** (how much to correct?).

In [ ]:
# Setup and Import Libraries
import os
import glob
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("✓ Libraries imported and device configured")

In [ ]:
# Data Loading Functions (copied from clean_point_cloud_analysis.ipynb)
def load_hdf5(dataset_path):
    """Load robotics episode data from HDF5 file with support for multiple action types."""
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None, None, None
    
    with h5py.File(dataset_path, 'r') as root:
        # Load joint data
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        
        # Load camera images
        image_dict = dict()
        if 'observations/images' in root:
            for cam_name in root[f'/observations/images/'].keys():
                image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        
        # Load all action types for QP filter analysis
        action_data = {}
        action_data['action'] = root['/action'][()]  # Final QP-filtered actions
        
        # Load additional action types if available
        if '/action_raw' in root:
            action_data['action_raw'] = root['/action_raw'][()]
        if '/action_processed' in root:
            action_data['action_processed'] = root['/action_processed'][()]
        if '/action_filtered' in root:
            action_data['action_filtered'] = root['/action_filtered'][()]
            
        # Load point cloud data if available
        pointcloud_dict = {}
        if 'observations/pointclouds' in root:
            for cam_name in root['observations/pointclouds'].keys():
                pointcloud_dict[cam_name] = {
                    'points_camera': root[f'observations/pointclouds/{cam_name}/points_camera'][()],
                    'points_world': root[f'observations/pointclouds/{cam_name}/points_world'][()], 
                    'num_points': root[f'observations/pointclouds/{cam_name}/num_points'][()]
                }
            
        # Load metadata attributes
        attrs = {}
        for key in root.attrs.keys():
            attrs[key] = root.attrs[key]
    
    return qpos, qvel, action_data, image_dict, attrs, pointcloud_dict

print("✓ Data loading functions ready")

In [ ]:
# Reactive PointNet Network Architecture (Much Deeper Version for Large Dataset)
class ReactivePointNet(nn.Module):
    """
    Much deeper PointNet-based reactive correction network for large-scale obstacle avoidance learning.
    
    Enhanced Deep Architecture for Large Dataset (episodes 0-128):
    - Input: Point cloud P ∈ R^{N×3} + joint state q ∈ R^6
    - Deep Encoder: Per-point MLP (3 → 64 → 128 → 256 → 512) + Multi-scale pooling → f_pc ∈ R^512
    - Multi-layer Fusion: [f_pc, q] → Deep MLP with multiple residual blocks
    - Output: Residual Δq̇ ∈ R^6 (arm joints only, no gripper)
    
    Key Enhancements for Large Dataset Learning:
    - 5-layer point encoder (vs 3-layer) for richer point features
    - 512D global features (vs 256D) for better representation capacity
    - Multiple residual blocks with skip connections for gradient flow
    - Dropout for regularization with large dataset
    - Batch normalization for training stability
    """
    
    def __init__(self, input_dim=3, joint_dim=6, output_dim=6, dropout_rate=0.1):
        super(ReactivePointNet, self).__init__()
        
        # Much deeper point cloud encoder: per-point MLP with more layers
        self.point_mlp = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(256, 512),  # Much deeper: added 4th layer
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(512, 512),  # Added 5th layer for even more capacity
            nn.BatchNorm1d(512),
            nn.ReLU()
        )
        
        # Enhanced fusion with multiple residual blocks
        fusion_input_dim = 512 + joint_dim  # 512 (point features) + 6 (joint state) = 518
        
        # Initial fusion layer
        self.fusion_input = nn.Sequential(
            nn.Linear(fusion_input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        # Multiple residual blocks for deep learning
        self.residual_block1 = self._make_residual_block(512, 512, dropout_rate)
        self.residual_block2 = self._make_residual_block(512, 512, dropout_rate)
        self.residual_block3 = self._make_residual_block(512, 256, dropout_rate)  # Reduce dimension
        
        # Final prediction layers with gradual dimension reduction
        self.prediction_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, output_dim)  # Final output: 6D residual
        )
        
        # Initialize weights
        self.apply(self._init_weights)
        
    def _make_residual_block(self, input_dim, output_dim, dropout_rate):
        """Create a residual block with batch norm and dropout."""
        return nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(output_dim, output_dim),
            nn.BatchNorm1d(output_dim)
        )
        
    def _init_weights(self, module):
        """Initialize network weights with proper scaling for deep networks."""
        if isinstance(module, nn.Linear):
            # Use He initialization for ReLU networks
            torch.nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.BatchNorm1d):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)
    
    def forward(self, points, joint_state):
        """
        Forward pass of the much deeper reactive network.
        
        Args:
            points: Point cloud tensor (B, N, 3) in robot frame
            joint_state: Joint positions (B, 6) for arm joints only
            
        Returns:
            residual: Predicted residual Δq̇ (B, 6)
        """
        batch_size, num_points, _ = points.shape
        
        # Reshape for batch normalization: (B, N, 3) → (B*N, 3)
        points_flat = points.view(-1, 3)
        
        # Encode each point through deep MLP: (B*N, 3) → (B*N, 512)
        point_features_flat = self.point_mlp(points_flat)
        
        # Reshape back: (B*N, 512) → (B, N, 512)
        point_features = point_features_flat.view(batch_size, num_points, -1)
        
        # Global max-pooling over points: (B, N, 512) → (B, 512)
        global_features, _ = torch.max(point_features, dim=1)
        
        # Alternative: Could also use attention-based pooling for even better performance
        # attention_weights = F.softmax(torch.mean(point_features, dim=2), dim=1)
        # global_features = torch.sum(point_features * attention_weights.unsqueeze(2), dim=1)
        
        # Concatenate with joint state: (B, 512) + (B, 6) → (B, 518)
        fused_features = torch.cat([global_features, joint_state], dim=1)
        
        # Initial fusion: (B, 518) → (B, 512)
        x = self.fusion_input(fused_features)
        
        # Multiple residual blocks with skip connections
        # Block 1: (B, 512) → (B, 512)
        residual_out1 = self.residual_block1(x)
        x = F.relu(x + residual_out1)  # Skip connection
        
        # Block 2: (B, 512) → (B, 512) 
        residual_out2 = self.residual_block2(x)
        x = F.relu(x + residual_out2)  # Skip connection
        
        # Block 3: (B, 512) → (B, 256) - dimension reduction
        residual_out3 = self.residual_block3(x)
        # Note: No skip connection here due to dimension change
        x = F.relu(residual_out3)
        
        # Final prediction through deep head: (B, 256) → (B, 6)
        residual = self.prediction_head(x)
        
        return residual

# Test the much deeper network architecture
def test_network():
    """Test network forward pass with dummy data."""
    print("🧪 Testing Much Deeper ReactivePointNet architecture...")
    
    # Create dummy data
    batch_size = 4
    num_points = 1500
    points = torch.randn(batch_size, num_points, 3).to(device)
    joint_state = torch.randn(batch_size, 6).to(device)
    
    # Initialize network
    model = ReactivePointNet(dropout_rate=0.1).to(device)
    
    # Forward pass
    with torch.no_grad():
        residual = model(points, joint_state)
    
    print(f"✓ Input shapes:")
    print(f"  Points: {points.shape}")
    print(f"  Joint state: {joint_state.shape}")
    print(f"✓ Output shape:")
    print(f"  Residual: {residual.shape}")
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"✓ Much deeper model parameters: {total_params:,}")
    print(f"✓ Trainable parameters: {trainable_params:,}")
    print(f"✓ Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB")
    
    # Architecture breakdown
    print(f"\n🏗️  Deep Architecture Details:")
    print(f"  Point encoder: 3→64→128→256→512→512 (5 layers with BN+Dropout)")
    print(f"  Global pooling: max-pool over N points → 512D global features")
    print(f"  Fusion input: [512 + 6] = 518D")
    print(f"  Fusion layers: 518→512→(3 residual blocks)→256→128→64→6")
    print(f"  Total depth: ~12 layers (5 encoder + 7 fusion)")
    print(f"  Regularization: Batch normalization + Dropout (0.1)")
    print(f"  Skip connections: Multiple residual blocks for gradient flow")
    
    print(f"\n📊 Capacity Comparison:")
    print(f"  Previous model: ~85K parameters")
    print(f"  New deep model: {total_params:,} parameters (~{total_params/85000:.1f}x larger)")
    print(f"  Memory usage: ~{total_params * 4 / 1024 / 1024:.1f} MB vs ~0.3 MB")
    
    return model

# Test the much deeper architecture
model = test_network()

In [ ]:
# Dataset Class for Reactive Training
class ReactiveDataset(torch.utils.data.Dataset):
    """
    PyTorch Dataset for reactive correction training.
    
    Loads point clouds, joint states, and computes target residuals from individual episode HDF5 files.
    target = action_processed - action_qp_filtered (QP filter correction)
    """
    
    def __init__(self, data_dir, sparsity_threshold=0.05):
        self.data_dir = data_dir
        self.sparsity_threshold = sparsity_threshold  # Make threshold configurable
        
        # Find all episode HDF5 files
        print(f"📂 Loading dataset from {data_dir}...")
        episode_files = glob.glob(os.path.join(data_dir, "episode_*.hdf5"))
        episode_files.sort(key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]))
        
        if not episode_files:
            raise FileNotFoundError(f"No episode files found in {data_dir}")
        
        # Cache episode information
        self.episode_files = episode_files
        self.episode_lengths = {}
        self.cumulative_lengths = [0]
        
        for ep_file in episode_files:
            ep_id = os.path.basename(ep_file).replace('.hdf5', '')
            
            with h5py.File(ep_file, 'r') as f:
                # Check for point clouds in the new structure
                if 'observations/pointclouds/front_depth/points_world' in f:
                    ep_len = len(f['observations/pointclouds/front_depth/points_world'])
                else:
                    print(f"⚠️  Warning: No point clouds found in {ep_file}")
                    continue
                
                self.episode_lengths[ep_id] = ep_len
                self.cumulative_lengths.append(self.cumulative_lengths[-1] + ep_len)
        
        self.total_length = self.cumulative_lengths[-1]
        self.valid_episodes = list(self.episode_lengths.keys())
        
        print(f"✓ Loaded {len(self.valid_episodes)} episodes")
        print(f"✓ Total timesteps: {self.total_length}")
        print(f"✓ Sparsity threshold: {self.sparsity_threshold:.3f} rad")
    
    def __len__(self):
        return self.total_length
    
    def _find_episode_and_index(self, idx):
        """Convert global index to (episode_file, local_index)."""
        # Find which episode contains this index
        episode_idx = 0
        for i, cum_len in enumerate(self.cumulative_lengths[1:]):
            if idx < cum_len:
                episode_idx = i
                break
        
        ep_id = self.valid_episodes[episode_idx]
        ep_file = os.path.join(self.data_dir, f"{ep_id}.hdf5")
        local_idx = idx - self.cumulative_lengths[episode_idx]
        return ep_file, local_idx
    
    def __getitem__(self, idx):
        """Get single training sample."""
        ep_file, local_idx = self._find_episode_and_index(idx)
        
        with h5py.File(ep_file, 'r') as f:
            # Load point cloud from new structure: (1500, 3)
            point_cloud = torch.from_numpy(
                f['observations/pointclouds/front_depth/points_world'][local_idx]
            ).float()
            
            # Load joint state (6,) - arm joints only
            joint_state = torch.from_numpy(
                f['observations/qpos'][local_idx][:6]
            ).float()
            
            # Load actions for residual computation (CORRECTED)
            action_qp_filtered = f['action'][local_idx]  # qdot_safe (QP filtered)
            action_processed = f['action_processed'][local_idx]  # qdot_nom (ACT commands)
            
            # Compute target residual: Δq̇ = action_processed - action_qp_filtered
            # This is what the QP filter "corrected" - we want to learn this directly
            target_residual = torch.from_numpy(action_processed[:6] - action_qp_filtered[:6]).float()
            
            # ADDED: Compute loss weight based on QP activation
            # When QP filter is active (large residual), weight = 1.0
            # When QP filter is inactive (small residual), weight = 0.2
            residual_norm = torch.norm(target_residual).item()
            loss_weight = 1.0 if residual_norm > self.sparsity_threshold else 0.2
        
        return {
            'points': point_cloud,          # (1500, 3)
            'joint_state': joint_state,     # (6,)
            'target_residual': target_residual,  # (6,)
            'loss_weight': torch.tensor(loss_weight, dtype=torch.float32)  # Scalar weight
        }

# Enhanced Training Configuration for Large Dataset and Deep Model
class TrainingConfig:
    """Enhanced training hyperparameters for large dataset and much deeper model."""
    
    # Data - Optimized for large dataset (episodes 0-128)
    batch_size = 16  # Reduced from 32 due to much larger model memory requirements
    num_workers = 6  # Increased for better data loading parallelism
    
    # Model - Optimized for deep network training
    learning_rate = 5e-4  # Reduced from 1e-3 for more stable deep network training
    weight_decay = 1e-4   # Keep same for regularization
    
    # Training - Extended for large dataset
    num_epochs = 100      # Increased from 50 for large dataset learning
    save_every = 20       # Save less frequently due to larger model
    validate_every = 5    # Keep same validation frequency
    
    # Learning rate schedule for deep network
    use_scheduler = True
    scheduler_patience = 10  # Reduce LR if no improvement for 10 epochs
    scheduler_factor = 0.5   # Reduce LR by half
    min_lr = 1e-6           # Minimum learning rate
    
    # Early stopping for large dataset training
    early_stopping_patience = 25  # Stop if no improvement for 25 epochs
    
    # Gradient clipping for deep network stability
    gradient_clip_norm = 1.0  # Clip gradients to prevent exploding gradients
    
    # Paths
    model_save_path = "reactive_pointnet_model.pth"
    checkpoint_dir = "checkpoints_deep/"
    
    # Loss weighting (if needed for different joints)
    joint_weights = torch.ones(6)  # Equal weighting for all joints
    
    # Sparsity weighting parameters
    sparsity_threshold = 0.05  # Threshold for QP activation detection
    active_weight = 1.0        # Weight for samples where QP is active
    inactive_weight = 0.2      # Weight for samples where QP is inactive (encourages sparsity)
    
    # Model architecture parameters
    dropout_rate = 0.1         # Dropout rate for regularization
    
    # Training monitoring
    log_interval = 100         # Log training progress every N batches
    
    # Mixed precision training for memory efficiency with large model
    use_mixed_precision = True  # Use automatic mixed precision (AMP)

config = TrainingConfig()
print(f"🔧 Enhanced Training Configuration for Large Dataset & Deep Model")
print(f"   Dataset: Episodes 0-128 (large scale)")
print(f"   Batch size: {config.batch_size} (reduced for deep model)")
print(f"   Learning rate: {config.learning_rate} (conservative for stability)")
print(f"   Epochs: {config.num_epochs} (extended for large dataset)")
print(f"   Workers: {config.num_workers}")
print(f"   Gradient clipping: {config.gradient_clip_norm}")
print(f"   Mixed precision: {config.use_mixed_precision}")
print(f"   Early stopping patience: {config.early_stopping_patience}")
print(f"🎯 Optimized for:")
print(f"   - Much deeper model (~500K+ parameters)")
print(f"   - Large dataset (episodes 39-128 + historical data)")  
print(f"   - Stable training with regularization")
print(f"   - Memory efficiency with mixed precision")
print(f"🔄 CORRECTED: Using action_processed - action_qp_filtered for residuals")

In [ ]:
# Load Dataset and Create Data Loaders
def create_data_loaders(data_dir, train_split=0.8, sparsity_threshold=0.05):
    """
    Create training and validation data loaders from individual episode HDF5 files.
    
    Args:
        data_dir: Directory containing episode_*.hdf5 files
        train_split: Fraction of data for training
        sparsity_threshold: Threshold for sparsity-aware loss weighting
    
    Returns:
        train_loader, val_loader, dataset_info
    """
    print(f"📊 Creating data loaders...")
    
    # Create full dataset from episode files with sparsity threshold
    full_dataset = ReactiveDataset(data_dir, sparsity_threshold=sparsity_threshold)
    
    # Split into train/validation by episodes (not individual timesteps)
    num_episodes = len(full_dataset.valid_episodes)
    num_train_episodes = int(num_episodes * train_split)
    
    train_episodes = full_dataset.valid_episodes[:num_train_episodes]
    val_episodes = full_dataset.valid_episodes[num_train_episodes:]
    
    print(f"✓ Train episodes: {len(train_episodes)} ({train_split:.1%})")
    print(f"✓ Val episodes: {len(val_episodes)} ({1-train_split:.1%})")
    
    # Create train/val indices based on episode splits
    train_indices = []
    val_indices = []
    
    current_idx = 0
    for ep_id in full_dataset.valid_episodes:
        ep_len = full_dataset.episode_lengths[ep_id]
        indices = list(range(current_idx, current_idx + ep_len))
        
        if ep_id in train_episodes:
            train_indices.extend(indices)
        else:
            val_indices.extend(indices)
            
        current_idx += ep_len
    
    # Create subset datasets
    train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
    val_dataset = torch.utils.data.Subset(full_dataset, val_indices)
    
    # Create data loaders
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=True
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=True
    )
    
    dataset_info = {
        'total_episodes': num_episodes,
        'train_episodes': len(train_episodes),
        'val_episodes': len(val_episodes),
        'train_samples': len(train_indices),
        'val_samples': len(val_indices),
        'total_samples': full_dataset.total_length,
        'sparsity_threshold': sparsity_threshold
    }
    
    print(f"✓ Training samples: {dataset_info['train_samples']:,}")
    print(f"✓ Validation samples: {dataset_info['val_samples']:,}")
    
    return train_loader, val_loader, dataset_info

# Load the data from individual episode HDF5 files
data_dir = "/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data"  # Directory containing episode_*.hdf5 files

try:
    # Use sparsity threshold from config
    train_loader, val_loader, dataset_info = create_data_loaders(
        data_dir, 
        sparsity_threshold=config.sparsity_threshold
    )
    print(f"\n🎯 Data loaders ready!")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")
    print(f"   Sparsity threshold: {dataset_info['sparsity_threshold']:.3f} rad")
except FileNotFoundError:
    print(f"❌ Error: Could not find episode files in {data_dir}")
    print("   Please ensure you have run the point cloud generation script first.")
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

In [ ]:
# Enhanced Training Loop Implementation for Deep Model
import time  # Add missing import

def weighted_mse_loss(predicted, target, weights):
    """
    Compute weighted MSE loss for sparsity-aware training.
    
    Args:
        predicted: Predicted residuals (B, 6)
        target: Target residuals (B, 6) 
        weights: Loss weights per sample (B,)
        
    Returns:
        Weighted MSE loss
    """
    # Compute per-sample MSE: (B, 6) -> (B,)
    mse_per_sample = torch.mean((predicted - target) ** 2, dim=1)
    
    # Apply weights: (B,) * (B,) -> (B,)
    weighted_mse = mse_per_sample * weights
    
    # Return mean weighted loss
    return torch.mean(weighted_mse)

def train_reactive_network_deep(model, train_loader, val_loader, config):
    """
    Enhanced training loop for much deeper ReactivePointNet with large dataset.
    
    Key enhancements for deep model training:
    - Mixed precision training for memory efficiency
    - Gradient clipping for stability
    - Advanced learning rate scheduling
    - Early stopping with patience
    - Comprehensive monitoring and logging
    
    Args:
        model: Much deeper ReactivePointNet model
        train_loader: Training data loader (large dataset)
        val_loader: Validation data loader
        config: Enhanced training configuration
    """
    
    # Setup training components with enhancements for deep networks
    optimizer = torch.optim.AdamW(  # AdamW for better regularization
        model.parameters(), 
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8
    )
    
    # Enhanced learning rate scheduler
    if config.use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 
            mode='min', 
            factor=config.scheduler_factor, 
            patience=config.scheduler_patience,
            min_lr=config.min_lr,
            verbose=True
        )
    
    # Mixed precision training for memory efficiency with large model
    scaler = None
    if config.use_mixed_precision:
        scaler = torch.cuda.amp.GradScaler()
        print("✓ Mixed precision training enabled for memory efficiency")
    
    # Create checkpoint directory
    os.makedirs(config.checkpoint_dir, exist_ok=True)
    
    # Training history with enhanced tracking
    history = {
        'train_loss': [],
        'val_loss': [],
        'learning_rate': [],
        'sparsity_stats': [],
        'gradient_norms': [],  # Track gradient norms for deep network monitoring
        'epoch_times': []      # Track training time per epoch
    }
    
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    
    print(f"🚀 Starting DEEP sparsity-aware training for {config.num_epochs} epochs")
    print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"   Model memory: ~{sum(p.numel() for p in model.parameters()) * 4 / 1024 / 1024:.1f} MB")
    print(f"   Device: {next(model.parameters()).device}")
    print(f"   Dataset size: Train={len(train_loader.dataset):,}, Val={len(val_loader.dataset):,}")
    print(f"   Advanced features: Mixed precision, Gradient clipping, Early stopping")
    
    for epoch in range(config.num_epochs):
        epoch_start_time = time.time()
        
        # Training phase with deep network enhancements
        model.train()
        train_loss = 0.0
        train_batches = 0
        active_samples = 0
        inactive_samples = 0
        gradient_norms = []
        
        for batch_idx, batch in enumerate(train_loader):
            # Move to device
            points = batch['points'].to(device)
            joint_state = batch['joint_state'].to(device)
            target_residual = batch['target_residual'].to(device)
            loss_weights = batch['loss_weight'].to(device)  # (B,)
            
            # Track sparsity statistics
            batch_size = points.shape[0]
            batch_active = torch.sum(loss_weights == config.active_weight).item()
            batch_inactive = batch_size - batch_active
            active_samples += batch_active
            inactive_samples += batch_inactive
            
            # Forward pass with mixed precision
            optimizer.zero_grad()
            
            if config.use_mixed_precision and scaler is not None:
                with torch.cuda.amp.autocast():
                    predicted_residual = model(points, joint_state)
                    loss = weighted_mse_loss(predicted_residual, target_residual, loss_weights)
                
                # Backward pass with gradient scaling
                scaler.scale(loss).backward()
                
                # Gradient clipping for deep network stability
                if config.gradient_clip_norm > 0:
                    scaler.unscale_(optimizer)
                    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
                    gradient_norms.append(total_norm.item())
                
                scaler.step(optimizer)
                scaler.update()
            else:
                # Standard precision training
                predicted_residual = model(points, joint_state)
                loss = weighted_mse_loss(predicted_residual, target_residual, loss_weights)
                loss.backward()
                
                # Gradient clipping
                if config.gradient_clip_norm > 0:
                    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
                    gradient_norms.append(total_norm.item())
                
                optimizer.step()
            
            # Accumulate loss
            train_loss += loss.item()
            train_batches += 1
            
            # Enhanced progress logging for large dataset
            if (batch_idx + 1) % config.log_interval == 0:
                active_pct = 100.0 * batch_active / batch_size
                current_lr = optimizer.param_groups[0]['lr']
                avg_grad_norm = np.mean(gradient_norms[-10:]) if gradient_norms else 0.0
                
                print(f"   Epoch {epoch+1:3d}/{config.num_epochs} | "
                      f"Batch {batch_idx+1:4d}/{len(train_loader)} | "
                      f"Loss: {loss.item():.6f} | "
                      f"Active: {active_pct:4.1f}% | "
                      f"LR: {current_lr:.2e} | "
                      f"GradNorm: {avg_grad_norm:.3f}")
        
        # Calculate epoch statistics
        avg_train_loss = train_loss / train_batches
        total_samples = active_samples + inactive_samples
        active_fraction = active_samples / total_samples if total_samples > 0 else 0.0
        avg_gradient_norm = np.mean(gradient_norms) if gradient_norms else 0.0
        epoch_time = time.time() - epoch_start_time
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['learning_rate'].append(optimizer.param_groups[0]['lr'])
        history['gradient_norms'].append(avg_gradient_norm)
        history['epoch_times'].append(epoch_time)
        history['sparsity_stats'].append({
            'active_samples': active_samples,
            'inactive_samples': inactive_samples,
            'active_fraction': active_fraction
        })
        
        # Validation phase
        val_loss = 0.0
        if (epoch + 1) % config.validate_every == 0:
            model.eval()
            val_batches = 0
            val_active_samples = 0
            val_inactive_samples = 0
            
            with torch.no_grad():
                for batch in val_loader:
                    points = batch['points'].to(device)
                    joint_state = batch['joint_state'].to(device)
                    target_residual = batch['target_residual'].to(device)
                    loss_weights = batch['loss_weight'].to(device)
                    
                    # Track validation sparsity
                    batch_size = points.shape[0]
                    batch_active = torch.sum(loss_weights == config.active_weight).item()
                    val_active_samples += batch_active
                    val_inactive_samples += batch_size - batch_active
                    
                    # Use mixed precision for validation too
                    if config.use_mixed_precision:
                        with torch.cuda.amp.autocast():
                            predicted_residual = model(points, joint_state)
                            loss = weighted_mse_loss(predicted_residual, target_residual, loss_weights)
                    else:
                        predicted_residual = model(points, joint_state)
                        loss = weighted_mse_loss(predicted_residual, target_residual, loss_weights)
                    
                    val_loss += loss.item()
                    val_batches += 1
            
            avg_val_loss = val_loss / val_batches
            val_total = val_active_samples + val_inactive_samples
            val_active_fraction = val_active_samples / val_total if val_total > 0 else 0.0
            
            history['val_loss'].append(avg_val_loss)
            
            # Update learning rate scheduler
            if config.use_scheduler:
                scheduler.step(avg_val_loss)
            
            # Early stopping and best model saving
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                epochs_without_improvement = 0
                
                # Save best model with enhanced metadata
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict() if config.use_scheduler else None,
                    'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
                    'train_loss': avg_train_loss,
                    'val_loss': avg_val_loss,
                    'config': config,
                    'history': history,
                    'model_architecture': 'ReactivePointNet_DeepV2',
                    'total_parameters': sum(p.numel() for p in model.parameters())
                }, config.model_save_path)
                print(f"   💾 New best model saved (val_loss: {avg_val_loss:.6f})")
            else:
                epochs_without_improvement += 1
            
            # Enhanced epoch summary
            print(f"   ✓ Epoch {epoch+1:3d} complete | "
                  f"Train: {avg_train_loss:.6f} | "
                  f"Val: {avg_val_loss:.6f} | "
                  f"Active: {100*active_fraction:.1f}%/{100*val_active_fraction:.1f}% | "
                  f"GradNorm: {avg_gradient_norm:.3f} | "
                  f"Time: {epoch_time:.1f}s | "
                  f"No-improve: {epochs_without_improvement}")
            
            # Early stopping check
            if epochs_without_improvement >= config.early_stopping_patience:
                print(f"\n🛑 Early stopping triggered after {epochs_without_improvement} epochs without improvement")
                break
        else:
            # No validation this epoch
            print(f"   ✓ Epoch {epoch+1:3d} complete | "
                  f"Train: {avg_train_loss:.6f} | "
                  f"Active: {100*active_fraction:.1f}% | "
                  f"GradNorm: {avg_gradient_norm:.3f} | "
                  f"Time: {epoch_time:.1f}s")
        
        # Save checkpoint
        if (epoch + 1) % config.save_every == 0:
            checkpoint_path = os.path.join(config.checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if config.use_scheduler else None,
                'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
                'history': history
            }, checkpoint_path)
    
    print(f"\n🎉 Deep model training completed!")
    print(f"   Best validation loss: {best_val_loss:.6f}")
    print(f"   Total training time: {sum(history['epoch_times']):.1f}s ({sum(history['epoch_times'])/60:.1f} min)")
    print(f"   Average epoch time: {np.mean(history['epoch_times']):.1f}s")
    print(f"   Model saved to: {config.model_save_path}")
    
    # Final comprehensive analysis
    final_stats = history['sparsity_stats'][-1]
    print(f"\n📊 Final Deep Model Statistics:")
    print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"   Active samples (QP engaged): {final_stats['active_samples']:,} ({100*final_stats['active_fraction']:.1f}%)")
    print(f"   Inactive samples (QP free): {final_stats['inactive_samples']:,} ({100*(1-final_stats['active_fraction']):.1f}%)")
    print(f"   Final gradient norm: {history['gradient_norms'][-1]:.3f}")
    print(f"   Final learning rate: {history['learning_rate'][-1]:.2e}")
    
    return history

# Initialize much deeper model and start enhanced training
print("🏗️  Initializing MUCH DEEPER ReactivePointNet model...")
model = ReactivePointNet(dropout_rate=config.dropout_rate).to(device)

print(f"📈 Deep Model Summary:")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB")
print(f"   Architecture: Deep PointNet(3→64→128→256→512→512) + Deep Fusion with residual blocks")
print(f"   Depth: ~12 layers with batch normalization and dropout")
print(f"   Memory requirement: ~{total_params * 4 / 1024 / 1024 * 2:.1f} MB (forward + backward)")

# Start enhanced training for large dataset
if 'train_loader' in locals() and 'val_loader' in locals():
    print(f"\n🎯 Starting DEEP sparsity-aware reactive network training...")
    print(f"   Dataset: {len(train_loader.dataset):,} training samples from episodes 0-128")
    print(f"   Model capacity: {total_params:,} parameters (~{total_params/85000:.1f}x larger than previous)")
    training_history = train_reactive_network_deep(model, train_loader, val_loader, config)
else:
    print("❌ Please run the data loading cell first!")

# 🎯 How Weighted Loss Teaches Sparsity: Simple Examples & Roadmap

## 🤔 **The Problem**: Standard Training vs Sparsity-Aware Training

### **Standard Training (BAD):**
```python
# Every sample gets same importance
loss = MSE(prediction, target)  # weight = 1.0 for all samples
```
**Result**: Model always predicts corrections, even when no obstacle is present! 🚫

### **Sparsity-Aware Training (GOOD):**
```python
# Different importance based on expert QP behavior
if expert_correction_norm > 0.05:  # QP was active
    loss_weight = 1.0  # "Learn this correction well!"
else:  # QP was inactive  
    loss_weight = 0.2  # "Learn to output near-zero"
```
**Result**: Model learns WHEN to activate, not just HOW MUCH to correct! ✅

---

## 🧠 **Simple Examples**

### **Example 1: No Obstacle (QP Inactive)**
```python
# Training data:
point_cloud = [...points far from robot...]
joint_state = [0.1, 0.5, -0.2, 0.0, 0.1, 0.0]
gt_residual = [0.001, 0.002, 0.000, 0.001, 0.000, 0.002]  # Nearly zero!
gt_norm = 0.003  # < threshold (0.05)

# Training:
loss_weight = 0.2  # Low weight - "don't worry too much about perfect precision"
loss = 0.2 * MSE(rn_prediction, gt_residual)
```
**Message to RN**: *"When you see this scene, output near-zero corrections"*

### **Example 2: Obstacle Present (QP Active)**
```python
# Training data:
point_cloud = [...points near robot gripper...]
joint_state = [0.2, 0.8, -0.5, 0.1, 0.2, 0.1]
gt_residual = [0.15, -0.08, 0.12, 0.05, -0.10, 0.03]  # Large corrections!
gt_norm = 0.23  # > threshold (0.05)

# Training:
loss_weight = 1.0  # High weight - "learn this correction precisely!"
loss = 1.0 * MSE(rn_prediction, gt_residual)
```
**Message to RN**: *"When you see obstacles, predict these exact corrections!"

---

## 🗺️ **Sparsity Learning Roadmap**

### **Phase 1: Data Understanding** 📊
```python
Expert QP Filter Analysis:
├── 80% of timesteps: QP inactive (||residual|| ≤ 0.05)
│   └── Target: RN should output ~0.01 corrections
├── 20% of timesteps: QP active (||residual|| > 0.05)  
│   └── Target: RN should match expert corrections exactly
└── Goal: Learn this 80/20 activation pattern
```

### **Phase 2: Weighted Training** 🎯
```python
Training Loop:
for batch in data_loader:
    # Standard prediction
    rn_prediction = model(points, joints)
    
    # Sparsity-aware weighting
    for sample in batch:
        if gt_norm[sample] > threshold:
            weight[sample] = 1.0    # "Critical: learn this!"
        else:
            weight[sample] = 0.2    # "Less important: stay small"
    
    # Weighted loss computation
    loss = weighted_mse_loss(rn_prediction, gt_residual, weights)
    loss.backward()  # Backprop with different gradients!
```

### **Phase 3: Gradient Impact** 📈
```python
Gradient Behavior:
├── QP Active samples (weight=1.0):
│   ├── Large gradients → Strong learning signal
│   ├── Model focuses on accuracy: "Match expert exactly!"
│   └── Result: RN learns precise corrections
├── QP Inactive samples (weight=0.2):
│   ├── Small gradients → Weak learning signal  
│   ├── Model learns: "Keep outputs small"
│   └── Result: RN outputs near-zero when safe
└── Combined Effect: Sparse activation pattern!
```

### **Phase 4: Learned Behavior** 🧠
```python
Trained RN Behavior:
├── Input: Point cloud + Joint state
├── Internal Decision:
│   ├── IF obstacles detected near trajectory:
│   │   └── Output: Large corrections (like expert QP)
│   └── ELSE (free space):
│       └── Output: Near-zero corrections  
└── Result: Sparse, expert-like activation! ✨
```

---

## 🎯 **Why This Works: Mathematical Intuition**

### **Standard Loss**: 
```python
L = (1/N) * Σ ||RN(x_i) - GT_i||²
```
**Problem**: All samples treated equally → Model averages between large and small corrections

### **Weighted Loss**:
```python
L = (1/N) * Σ w_i * ||RN(x_i) - GT_i||²
where: w_i = 1.0 if ||GT_i|| > 0.05 else 0.2
```
**Solution**: Important samples get 5x more influence → Model learns when to activate!

### **Training Dynamics**:
```python
Gradient Flow:
├── Active samples: ∇L_active = 1.0 * 2 * (RN - GT) * ∇RN
│   └── Strong signal: "Adjust weights to match GT precisely!"
├── Inactive samples: ∇L_inactive = 0.2 * 2 * (RN - GT) * ∇RN  
│   └── Weak signal: "Gently push towards zero"
└── Net effect: Model learns sparse activation pattern
```

---

## 🚀 **Expected Training Progression**

### **Early Training (Epochs 1-10)**:
- Model outputs random corrections everywhere
- High loss on both active and inactive samples
- Gradual learning of basic point cloud → residual mapping

### **Mid Training (Epochs 10-30)**:
- Model starts outputting smaller corrections for free space
- Better accuracy on obstacle avoidance corrections  
- Sparsity pattern begins to emerge

### **Late Training (Epochs 30-50)**:
- Model achieves sparse activation: ~0.01 for free space, ~0.2 for obstacles
- High correlation with expert QP activation timing
- Ready for deployment! 🎉

This weighted training creates a **smart, sparse reactive network** that knows both **when** and **how** to intervene! 🧠✨

In [ ]:
# Visualization and Evaluation Functions
def plot_training_history(history):
    """Plot training and validation loss curves with sparsity statistics."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 8))
    
    # Loss curves
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    
    # Validation loss (only on validation epochs)
    if history['val_loss']:
        val_epochs = range(config.validate_every, len(history['train_loss']) + 1, config.validate_every)
        axes[0, 0].plot(val_epochs, history['val_loss'], 'r-', label='Validation Loss')
    
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('MSE Loss')
    axes[0, 0].set_title('Training Progress')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Learning rate
    axes[0, 1].plot(epochs, history['learning_rate'], 'g-', label='Learning Rate')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_title('Learning Rate Schedule')
    axes[0, 1].set_yscale('log')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Sparsity statistics - QP activation frequency
    if 'sparsity_stats' in history:
        active_fractions = [stat['active_fraction'] for stat in history['sparsity_stats']]
        axes[1, 0].plot(epochs, active_fractions, 'purple', label='QP Activation Rate')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Fraction of Samples')
        axes[1, 0].set_title('QP Filter Activation Frequency')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].set_ylim(0, 1)
        
        # Sample counts
        active_counts = [stat['active_samples'] for stat in history['sparsity_stats']]
        inactive_counts = [stat['inactive_samples'] for stat in history['sparsity_stats']]
        
        axes[1, 1].plot(epochs, active_counts, 'orange', label='QP Active Samples')
        axes[1, 1].plot(epochs, inactive_counts, 'cyan', label='QP Inactive Samples')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Sample Count')
        axes[1, 1].set_title('Sample Distribution per Epoch')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def evaluate_model_predictions(model, val_loader, num_samples=100):
    """
    Evaluate model predictions with sparsity-aware analysis.
    
    Args:
        model: Trained ReactivePointNet model
        val_loader: Validation data loader
        num_samples: Number of samples to evaluate
    """
    model.eval()
    
    predictions = []
    targets = []
    weights = []
    
    print(f"🔍 Evaluating model on {num_samples} validation samples...")
    
    with torch.no_grad():
        sample_count = 0
        for batch in val_loader:
            if sample_count >= num_samples:
                break
                
            points = batch['points'].to(device)
            joint_state = batch['joint_state'].to(device)
            target_residual = batch['target_residual']
            loss_weights = batch['loss_weight']
            
            predicted_residual = model(points, joint_state).cpu()
            
            # Collect predictions and targets
            batch_size = points.shape[0]
            for i in range(min(batch_size, num_samples - sample_count)):
                predictions.append(predicted_residual[i].numpy())
                targets.append(target_residual[i].numpy())
                weights.append(loss_weights[i].numpy())
                sample_count += 1
    
    predictions = np.array(predictions)  # (N, 6)
    targets = np.array(targets)          # (N, 6)
    weights = np.array(weights)          # (N,)
    
    # Separate active vs inactive samples
    active_mask = weights == config.active_weight
    inactive_mask = weights == config.inactive_weight
    
    print(f"\n📊 Sparsity Analysis (n={len(predictions)}):")
    print(f"   QP Active samples: {np.sum(active_mask)} ({100*np.mean(active_mask):.1f}%)")
    print(f"   QP Inactive samples: {np.sum(inactive_mask)} ({100*np.mean(inactive_mask):.1f}%)")
    
    # Compute statistics for all samples
    mse_per_joint = np.mean((predictions - targets) ** 2, axis=0)
    mae_per_joint = np.mean(np.abs(predictions - targets), axis=0)
    
    # Compute statistics for active vs inactive samples
    if np.sum(active_mask) > 0:
        active_mse = np.mean((predictions[active_mask] - targets[active_mask]) ** 2, axis=0)
        active_mae = np.mean(np.abs(predictions[active_mask] - targets[active_mask]), axis=0)
        active_target_norm = np.mean(np.linalg.norm(targets[active_mask], axis=1))
        active_pred_norm = np.mean(np.linalg.norm(predictions[active_mask], axis=1))
    else:
        active_mse = active_mae = active_target_norm = active_pred_norm = 0.0
    
    if np.sum(inactive_mask) > 0:
        inactive_mse = np.mean((predictions[inactive_mask] - targets[inactive_mask]) ** 2, axis=0)
        inactive_mae = np.mean(np.abs(predictions[inactive_mask] - targets[inactive_mask]), axis=0)
        inactive_target_norm = np.mean(np.linalg.norm(targets[inactive_mask], axis=1))
        inactive_pred_norm = np.mean(np.linalg.norm(predictions[inactive_mask], axis=1))
    else:
        inactive_mse = inactive_mae = inactive_target_norm = inactive_pred_norm = 0.0
    
    joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6']
    
    print(f"\n📊 Overall Model Performance (n={len(predictions)}):")
    print(f"{'Joint':<8} {'MSE':<10} {'MAE':<10} {'Target Std':<12} {'Pred Std':<12}")
    print("-" * 60)
    
    for i, joint_name in enumerate(joint_names):
        target_std = np.std(targets[:, i])
        pred_std = np.std(predictions[:, i])
        print(f"{joint_name:<8} {mse_per_joint[i]:<10.6f} {mae_per_joint[i]:<10.6f} "
              f"{target_std:<12.6f} {pred_std:<12.6f}")
    
    # Sparsity-specific performance
    print(f"\n🎯 Sparsity-Aware Performance:")
    print(f"   QP Active (n={np.sum(active_mask)}):")
    print(f"     Mean residual norm - Target: {active_target_norm:.6f}, Predicted: {active_pred_norm:.6f}")
    if np.sum(active_mask) > 0:
        print(f"     Mean MSE: {np.mean(active_mse):.6f}")
        print(f"     Mean MAE: {np.mean(active_mae):.6f}")
    
    print(f"   QP Inactive (n={np.sum(inactive_mask)}):")
    print(f"     Mean residual norm - Target: {inactive_target_norm:.6f}, Predicted: {inactive_pred_norm:.6f}")
    if np.sum(inactive_mask) > 0:
        print(f"     Mean MSE: {np.mean(inactive_mse):.6f}")
        print(f"     Mean MAE: {np.mean(inactive_mae):.6f}")
    
    overall_mse = np.mean(mse_per_joint)
    overall_mae = np.mean(mae_per_joint)
    
    print(f"\n🎯 Overall Performance:")
    print(f"   Mean MSE: {overall_mse:.6f}")
    print(f"   Mean MAE: {overall_mae:.6f}")
    
    # Check sparsity learning success
    sparsity_success = inactive_pred_norm < 0.02  # Model outputs near-zero for inactive samples
    precision_success = active_pred_norm > 0.02   # Model outputs meaningful corrections for active samples
    
    print(f"\n🎯 Sparsity Learning Assessment:")
    print(f"   ✅ Inactive sparsity: {sparsity_success} (pred norm: {inactive_pred_norm:.6f})")
    print(f"   ✅ Active precision: {precision_success} (pred norm: {active_pred_norm:.6f})")
    
    # Plot comparison with sparsity awareness
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, joint_name in enumerate(joint_names):
        ax = axes[i]
        
        # Scatter plot with different colors for active/inactive
        if np.sum(active_mask) > 0:
            ax.scatter(targets[active_mask, i], predictions[active_mask, i], 
                      alpha=0.7, s=15, c='red', label='QP Active')
        if np.sum(inactive_mask) > 0:
            ax.scatter(targets[inactive_mask, i], predictions[inactive_mask, i], 
                      alpha=0.7, s=15, c='blue', label='QP Inactive')
        
        # Perfect prediction line
        min_val = min(np.min(targets[:, i]), np.min(predictions[:, i]))
        max_val = max(np.max(targets[:, i]), np.max(predictions[:, i]))
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.8, label='Perfect')
        
        ax.set_xlabel('Target Residual')
        ax.set_ylabel('Predicted Residual')
        ax.set_title(f'{joint_name} (MSE: {mse_per_joint[i]:.6f})')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'predictions': predictions,
        'targets': targets,
        'weights': weights,
        'mse_per_joint': mse_per_joint,
        'mae_per_joint': mae_per_joint,
        'overall_mse': overall_mse,
        'overall_mae': overall_mae,
        'active_performance': {
            'mse': active_mse if np.sum(active_mask) > 0 else None,
            'mae': active_mae if np.sum(active_mask) > 0 else None,
            'pred_norm': active_pred_norm,
            'target_norm': active_target_norm
        },
        'inactive_performance': {
            'mse': inactive_mse if np.sum(inactive_mask) > 0 else None,
            'mae': inactive_mae if np.sum(inactive_mask) > 0 else None,
            'pred_norm': inactive_pred_norm,
            'target_norm': inactive_target_norm
        }
    }

def load_trained_model(model_path):
    """Load a trained model from checkpoint."""
    if not os.path.exists(model_path):
        print(f"❌ Model file not found: {model_path}")
        return None
    
    print(f"📥 Loading trained model from {model_path}...")
    
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    
    # Initialize model and load weights
    model = ReactivePointNet().to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print(f"✓ Model loaded successfully!")
    print(f"   Epoch: {checkpoint['epoch']}")
    print(f"   Training loss: {checkpoint['train_loss']:.6f}")
    if 'val_loss' in checkpoint:
        print(f"   Validation loss: {checkpoint['val_loss']:.6f}")
    
    return model

# Evaluation helper
def quick_evaluation():
    """Quick evaluation of the trained model."""
    if 'model' in locals() and 'val_loader' in locals():
        print("🔍 Running quick model evaluation...")
        eval_results = evaluate_model_predictions(model, val_loader, num_samples=200)
        return eval_results
    else:
        print("❌ Please train the model first or load a trained checkpoint!")

print("📊 Sparsity-aware evaluation functions ready!")
print("   - plot_training_history(history): Plot training curves + sparsity stats")
print("   - evaluate_model_predictions(model, val_loader): Detailed sparsity-aware evaluation")
print("   - load_trained_model(path): Load trained checkpoint")
print("   - quick_evaluation(): Quick evaluation of current model")

In [ ]:
# 🔍 Model Evaluation - Run this cell to evaluate your trained reactive PointNet!

print("🎯 Starting comprehensive model evaluation...")
print("=" * 60)

# 1. Plot training curves to see learning progress
print("\n📈 1. Training History Analysis:")
if 'training_history' in locals():
    plot_training_history(training_history)
else:
    print("   ⚠️  Training history not available. Model was loaded from checkpoint.")

# 2. Evaluate model predictions on validation set
print("\n📊 2. Model Performance on Validation Set:")
eval_results = evaluate_model_predictions(model, val_loader, num_samples=500)

# 3. Analyze per-joint performance
print("\n🎯 3. Per-Joint Residual Analysis:")
joint_names = ['Shoulder', 'Shoulder Lift', 'Elbow', 'Wrist 1', 'Wrist 2', 'Wrist 3']

print(f"\n📋 Residual Prediction Quality Summary:")
print(f"{'Joint':<15} {'MSE':<12} {'MAE':<12} {'Quality':<10}")
print("-" * 55)

for i, joint_name in enumerate(joint_names):
    mse = eval_results['mse_per_joint'][i]
    mae = eval_results['mae_per_joint'][i]
    
    # Quality assessment based on MSE
    if mse < 0.001:
        quality = "Excellent"
    elif mse < 0.01:
        quality = "Good"
    elif mse < 0.1:
        quality = "Fair"
    else:
        quality = "Poor"
    
    print(f"{joint_name:<15} {mse:<12.6f} {mae:<12.6f} {quality:<10}")

print(f"\n🎯 Overall Assessment:")
print(f"   Mean MSE: {eval_results['overall_mse']:.6f}")
print(f"   Mean MAE: {eval_results['overall_mae']:.6f}")

if eval_results['overall_mse'] < 0.01:
    print(f"   📊 Status: ✅ EXCELLENT - Ready for deployment!")
elif eval_results['overall_mse'] < 0.1:
    print(f"   📊 Status: ✅ GOOD - Suitable for testing")
else:
    print(f"   📊 Status: ⚠️  NEEDS IMPROVEMENT - Consider more training")

# 4. Sample predictions analysis
print(f"\n🔬 4. Sample Predictions Analysis:")
print(f"   Validation samples analyzed: {len(eval_results['predictions'])}")
print(f"   Point cloud input: (1500, 3) per sample")
print(f"   Joint state input: (6,) per sample") 
print(f"   Residual output: (6,) per sample")

# 5. Model architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n🏗️  5. Model Architecture Summary:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB (float32)")
print(f"   Architecture: PointNet(3→64→128) + Fusion(134→128→64→6)")

print(f"\n✅ Evaluation complete! Your reactive PointNet is ready for obstacle avoidance testing.")
print(f"💡 Next steps:")
print(f"   - Test in simulation with ACT + reactive corrections")
print(f"   - Compare performance: ACT-only vs ACT+QP vs ACT+PointNet")
print(f"   - Deploy for real-world reactive obstacle avoidance")

In [ ]:
# 🔬 Episode-Level RN vs GT Residual Comparison
def compare_rn_vs_gt_episode(model, data_dir, episode_id=None, max_timesteps=200):
    """
    Compare Reactive Network predictions vs Ground Truth residuals for a full episode.
    
    Args:
        model: Trained ReactivePointNet model
        data_dir: Directory containing episode HDF5 files
        episode_id: Specific episode to analyze (e.g., "episode_0"), if None uses first episode
        max_timesteps: Maximum timesteps to analyze for visualization
    """
    
    # Find available episodes
    episode_files = glob.glob(os.path.join(data_dir, "episode_*.hdf5"))
    episode_files.sort(key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]))
    
    if not episode_files:
        print(f"❌ No episode files found in {data_dir}")
        return
    
    # Select episode
    if episode_id is None:
        episode_file = episode_files[24]  # Use first episode
        episode_id = os.path.basename(episode_file).replace('.hdf5', '')
    else:
        episode_file = os.path.join(data_dir, f"{episode_id}.hdf5")
        if not os.path.exists(episode_file):
            print(f"❌ Episode file not found: {episode_file}")
            return
    
    print(f"🔬 Analyzing episode: {episode_id}")
    print(f"📁 File: {episode_file}")
    
    model.eval()
    
    # Load episode data
    with h5py.File(episode_file, 'r') as f:
        # Check data availability
        if 'observations/pointclouds/front_depth/points_world' not in f:
            print(f"❌ No point cloud data found in {episode_file}")
            return
            
        # Load episode data
        points_world = f['observations/pointclouds/front_depth/points_world'][()]
        joint_states = f['observations/qpos'][()][:, :6]  # First 6 joints only
        action_qp_filtered = f['action'][()]  # qdot_safe
        action_processed = f['action_processed'][()]  # qdot_nom
        
        # Get episode metadata
        obstacle_height = f.attrs.get('obstacle_height', 'unknown')
        qp_enabled = f.attrs.get('qp_filter_enabled', 'unknown')
        
        episode_length = len(points_world)
        analyze_length = min(episode_length, max_timesteps)
        
    print(f"📊 Episode Info:")
    print(f"   Length: {episode_length} timesteps")
    print(f"   Analyzing: {analyze_length} timesteps")
    print(f"   Obstacle height: {obstacle_height}")
    print(f"   QP filter enabled: {qp_enabled}")
    
    # Compute GT residuals and RN predictions
    gt_residuals = []
    rn_predictions = []
    qp_active_flags = []
    
    print(f"🧠 Computing RN predictions vs GT residuals...")
    
    with torch.no_grad():
        for t in range(analyze_length):
            # Ground truth residual (what QP filter actually corrected)
            gt_residual = action_processed[t][:6] - action_qp_filtered[t][:6]
            gt_residuals.append(gt_residual)
            
            # Reactive Network prediction
            points_tensor = torch.from_numpy(points_world[t]).float().unsqueeze(0).to(device)  # (1, 1500, 3)
            joint_tensor = torch.from_numpy(joint_states[t]).float().unsqueeze(0).to(device)   # (1, 6)
            
            rn_residual = model(points_tensor, joint_tensor).cpu().numpy()[0]  # (6,)
            rn_predictions.append(rn_residual)
            
            # Check if QP was active (residual norm > threshold)
            gt_norm = np.linalg.norm(gt_residual)
            qp_active = gt_norm > config.sparsity_threshold
            qp_active_flags.append(qp_active)
    
    gt_residuals = np.array(gt_residuals)      # (T, 6)
    rn_predictions = np.array(rn_predictions)  # (T, 6)
    qp_active_flags = np.array(qp_active_flags)  # (T,)
    
    # Compute statistics
    mse_per_joint = np.mean((rn_predictions - gt_residuals) ** 2, axis=0)
    mae_per_joint = np.mean(np.abs(rn_predictions - gt_residuals), axis=0)
    
    gt_norms = np.linalg.norm(gt_residuals, axis=1)
    rn_norms = np.linalg.norm(rn_predictions, axis=1)
    
    qp_active_count = np.sum(qp_active_flags)
    qp_inactive_count = len(qp_active_flags) - qp_active_count
    
    print(f"\n📊 Episode Analysis Results:")
    print(f"   QP Active timesteps: {qp_active_count}/{analyze_length} ({100*qp_active_count/analyze_length:.1f}%)")
    print(f"   QP Inactive timesteps: {qp_inactive_count}/{analyze_length} ({100*qp_inactive_count/analyze_length:.1f}%)")
    print(f"   Mean GT residual norm: {np.mean(gt_norms):.6f}")
    print(f"   Mean RN residual norm: {np.mean(rn_norms):.6f}")
    print(f"   Overall MSE: {np.mean(mse_per_joint):.6f}")
    print(f"   Overall MAE: {np.mean(mae_per_joint):.6f}")
    
    # Separate active vs inactive performance
    if qp_active_count > 0:
        active_gt_norm = np.mean(gt_norms[qp_active_flags])
        active_rn_norm = np.mean(rn_norms[qp_active_flags])
        active_mse = np.mean((rn_predictions[qp_active_flags] - gt_residuals[qp_active_flags]) ** 2)
        print(f"   QP Active - GT norm: {active_gt_norm:.6f}, RN norm: {active_rn_norm:.6f}, MSE: {active_mse:.6f}")
    
    if qp_inactive_count > 0:
        inactive_gt_norm = np.mean(gt_norms[~qp_active_flags])
        inactive_rn_norm = np.mean(rn_norms[~qp_active_flags])
        inactive_mse = np.mean((rn_predictions[~qp_active_flags] - gt_residuals[~qp_active_flags]) ** 2)
        print(f"   QP Inactive - GT norm: {inactive_gt_norm:.6f}, RN norm: {inactive_rn_norm:.6f}, MSE: {inactive_mse:.6f}")
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Time series comparison of residual norms
    ax1 = plt.subplot(3, 3, 1)
    timesteps = np.arange(analyze_length)
    ax1.plot(timesteps, gt_norms, 'b-', label='GT Residual Norm', alpha=0.8, linewidth=2)
    ax1.plot(timesteps, rn_norms, 'r--', label='RN Prediction Norm', alpha=0.8, linewidth=2)
    ax1.axhline(y=config.sparsity_threshold, color='gray', linestyle=':', label=f'Sparsity Threshold ({config.sparsity_threshold})')
    
    # Highlight QP active regions
    for t in range(analyze_length):
        if qp_active_flags[t]:
            ax1.axvspan(t-0.5, t+0.5, alpha=0.2, color='yellow', zorder=0)
    
    ax1.set_xlabel('Timestep')
    ax1.set_ylabel('Residual Norm (rad)')
    ax1.set_title(f'Episode {episode_id}: Residual Magnitude Comparison')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2-7. Per-joint comparisons
    joint_names = ['Joint 1', 'Joint 2', 'Joint 3', 'Joint 4', 'Joint 5', 'Joint 6']
    
    for i in range(6):
        ax = plt.subplot(3, 3, i+2)
        ax.plot(timesteps, gt_residuals[:, i], 'b-', label='GT', alpha=0.8, linewidth=1.5)
        ax.plot(timesteps, rn_predictions[:, i], 'r--', label='RN', alpha=0.8, linewidth=1.5)
        
        # Highlight QP active regions
        for t in range(analyze_length):
            if qp_active_flags[t]:
                ax.axvspan(t-0.5, t+0.5, alpha=0.2, color='yellow', zorder=0)
        
        ax.set_xlabel('Timestep')
        ax.set_ylabel('Residual (rad/s)')
        ax.set_title(f'{joint_names[i]} (MSE: {mse_per_joint[i]:.6f})')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # 8. Correlation scatter plot
    ax8 = plt.subplot(3, 3, 8)
    
    # Separate colors for active/inactive
    if qp_active_count > 0:
        ax8.scatter(gt_norms[qp_active_flags], rn_norms[qp_active_flags], 
                   c='red', alpha=0.7, s=30, label=f'QP Active ({qp_active_count})')
    if qp_inactive_count > 0:
        ax8.scatter(gt_norms[~qp_active_flags], rn_norms[~qp_active_flags], 
                   c='blue', alpha=0.7, s=30, label=f'QP Inactive ({qp_inactive_count})')
    
    # Perfect correlation line
    max_norm = max(np.max(gt_norms), np.max(rn_norms))
    ax8.plot([0, max_norm], [0, max_norm], 'k--', alpha=0.8, label='Perfect')
    
    ax8.set_xlabel('GT Residual Norm')
    ax8.set_ylabel('RN Prediction Norm')
    ax8.set_title('Residual Norm Correlation')
    ax8.legend()
    ax8.grid(True, alpha=0.3)
    
    # 9. Error distribution
    ax9 = plt.subplot(3, 3, 9)
    
    errors = np.linalg.norm(rn_predictions - gt_residuals, axis=1)
    ax9.hist(errors, bins=30, alpha=0.7, color='purple', edgecolor='black')
    ax9.axvline(x=np.mean(errors), color='red', linestyle='--', label=f'Mean Error: {np.mean(errors):.6f}')
    ax9.set_xlabel('Prediction Error Norm')
    ax9.set_ylabel('Frequency')
    ax9.set_title('Error Distribution')
    ax9.legend()
    ax9.grid(True, alpha=0.3)
    
    plt.suptitle(f'Reactive Network vs Ground Truth: {episode_id} (Obstacle: {obstacle_height}m)', fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Return results for further analysis
    return {
        'episode_id': episode_id,
        'gt_residuals': gt_residuals,
        'rn_predictions': rn_predictions,
        'qp_active_flags': qp_active_flags,
        'mse_per_joint': mse_per_joint,
        'mae_per_joint': mae_per_joint,
        'obstacle_height': obstacle_height,
        'qp_enabled': qp_enabled,
        'stats': {
            'qp_active_count': qp_active_count,
            'qp_inactive_count': qp_inactive_count,
            'mean_gt_norm': np.mean(gt_norms),
            'mean_rn_norm': np.mean(rn_norms),
            'overall_mse': np.mean(mse_per_joint),
            'overall_mae': np.mean(mae_per_joint)
        }
    }

# Run the episode comparison
print("🔬 Episode-Level Reactive Network Analysis")
print("=" * 60)

# Check if model and data are available
if 'model' in locals() and 'data_dir' in locals():
    # Analyze the first episode by default
    episode_results = compare_rn_vs_gt_episode(model, data_dir, episode_id=None, max_timesteps=150)
    
    print(f"\n💡 Analysis Summary:")
    print(f"   Episode: {episode_results['episode_id']}")
    print(f"   QP Activation Rate: {100*episode_results['stats']['qp_active_count']/(episode_results['stats']['qp_active_count']+episode_results['stats']['qp_inactive_count']):.1f}%")
    print(f"   Prediction Accuracy: MSE = {episode_results['stats']['overall_mse']:.6f}")
    
    if episode_results['stats']['overall_mse'] < 0.01:
        print(f"   🎯 Status: ✅ EXCELLENT prediction accuracy!")
    elif episode_results['stats']['overall_mse'] < 0.1:
        print(f"   🎯 Status: ✅ GOOD prediction accuracy")
    else:
        print(f"   🎯 Status: ⚠️ Could benefit from more training")
        
else:
    print("❌ Please ensure model is trained/loaded and dataset is available!")
    print("   Run the training cells or load a trained model first.")

In [ ]:
# Deep Model Training Visualization and Analysis
import matplotlib.pyplot as plt
import numpy as np

def plot_deep_training_analysis(history, save_path='deep_training_analysis.png'):
    """
    Comprehensive visualization of deep model training progress.
    
    Shows training loss, validation loss, learning rate, gradient norms,
    sparsity statistics, and epoch timing for the much deeper model.
    """
    
    if not history or not history['train_loss']:
        print("❌ No training history available. Please train the model first.")
        return
    
    # Create comprehensive analysis figure
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    fig.suptitle('Deep ReactivePointNet Training Analysis\n~500K+ Parameters | Episodes 0-128', 
                 fontsize=16, fontweight='bold')
    
    epochs = range(1, len(history['train_loss']) + 1)
    val_epochs = [i * config.validate_every for i in range(1, len(history['val_loss']) + 1)]
    
    # 1. Loss curves with enhanced visualization
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Training Loss', alpha=0.8)
    if history['val_loss']:
        axes[0, 0].plot(val_epochs, history['val_loss'], 'r-', linewidth=2, label='Validation Loss', alpha=0.8)
        
        # Mark best validation loss
        best_val_idx = np.argmin(history['val_loss'])
        best_val_loss = history['val_loss'][best_val_idx]
        best_val_epoch = val_epochs[best_val_idx]
        axes[0, 0].plot(best_val_epoch, best_val_loss, 'go', markersize=8, 
                       label=f'Best Val: {best_val_loss:.6f}')
    
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Weighted MSE Loss')
    axes[0, 0].set_title('Deep Model Loss Curves')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_yscale('log')
    
    # 2. Learning rate progression
    axes[0, 1].plot(epochs, history['learning_rate'], 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_title('Learning Rate Schedule\n(AdamW + ReduceLROnPlateau)')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_yscale('log')
    
    # 3. Gradient norm tracking (stability indicator for deep network)
    if history['gradient_norms']:
        axes[1, 0].plot(epochs, history['gradient_norms'], 'purple', linewidth=2)
        axes[1, 0].axhline(y=config.gradient_clip_norm, color='red', linestyle='--', alpha=0.7,
                          label=f'Clip threshold: {config.gradient_clip_norm}')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Average Gradient Norm')
        axes[1, 0].set_title('Gradient Norms\n(Deep Network Stability)')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Sparsity analysis over training
    if history['sparsity_stats']:
        active_fractions = [stat['active_fraction'] for stat in history['sparsity_stats']]
        active_counts = [stat['active_samples'] for stat in history['sparsity_stats']]
        
        ax_sparse = axes[1, 1]
        ax_count = ax_sparse.twinx()
        
        line1 = ax_sparse.plot(epochs, [100 * af for af in active_fractions], 'orange', 
                              linewidth=2, label='Active %')
        line2 = ax_count.plot(epochs, active_counts, 'brown', linewidth=2, 
                             label='Active Count', alpha=0.7)
        
        ax_sparse.set_xlabel('Epoch')
        ax_sparse.set_ylabel('Active Samples (%)', color='orange')
        ax_count.set_ylabel('Active Sample Count', color='brown')
        ax_sparse.set_title('Dataset Sparsity Analysis\n(QP Engagement)')
        
        # Combine legends
        lines1, labels1 = ax_sparse.get_legend_handles_labels()
        lines2, labels2 = ax_count.get_legend_handles_labels()
        ax_sparse.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        
        ax_sparse.grid(True, alpha=0.3)
    
    # 5. Epoch timing analysis
    if history['epoch_times']:
        axes[2, 0].plot(epochs, history['epoch_times'], 'teal', linewidth=2)
        axes[2, 0].axhline(y=np.mean(history['epoch_times']), color='red', linestyle='--', 
                          alpha=0.7, label=f'Average: {np.mean(history["epoch_times"]):.1f}s')
        axes[2, 0].set_xlabel('Epoch')
        axes[2, 0].set_ylabel('Epoch Time (seconds)')
        axes[2, 0].set_title('Training Time per Epoch\n(Deep Model Performance)')
        axes[2, 0].legend()
        axes[2, 0].grid(True, alpha=0.3)
    
    # 6. Comprehensive training statistics summary
    axes[2, 1].axis('off')
    
    # Calculate comprehensive statistics
    total_epochs = len(history['train_loss'])
    final_train_loss = history['train_loss'][-1]
    final_val_loss = history['val_loss'][-1] if history['val_loss'] else 'N/A'
    total_time = sum(history['epoch_times']) if history['epoch_times'] else 0
    avg_epoch_time = np.mean(history['epoch_times']) if history['epoch_times'] else 0
    
    final_sparsity = history['sparsity_stats'][-1] if history['sparsity_stats'] else None
    active_pct = 100 * final_sparsity['active_fraction'] if final_sparsity else 0
    
    final_grad_norm = history['gradient_norms'][-1] if history['gradient_norms'] else 'N/A'
    final_lr = history['learning_rate'][-1] if history['learning_rate'] else 'N/A'
    
    # Improvement metrics
    if len(history['train_loss']) > 1:
        train_improvement = ((history['train_loss'][0] - history['train_loss'][-1]) / 
                           history['train_loss'][0] * 100)
    else:
        train_improvement = 0
    
    if len(history['val_loss']) > 1:
        val_improvement = ((history['val_loss'][0] - history['val_loss'][-1]) / 
                         history['val_loss'][0] * 100)
        best_val = min(history['val_loss'])
    else:
        val_improvement = 0
        best_val = 'N/A'
    
    # Create detailed statistics text
    stats_text = f"""DEEP MODEL TRAINING SUMMARY
    
Architecture: ReactivePointNet Deep V2
Parameters: {sum(p.numel() for p in model.parameters()):,} (~500K+)
Dataset: Episodes 0-128 (Large Scale)

TRAINING RESULTS:
✓ Total Epochs: {total_epochs}
✓ Final Train Loss: {final_train_loss:.6f}
✓ Final Val Loss: {final_val_loss if isinstance(final_val_loss, str) else f'{final_val_loss:.6f}'}
✓ Best Val Loss: {best_val if isinstance(best_val, str) else f'{best_val:.6f}'}

IMPROVEMENTS:
↗ Train Loss: {train_improvement:.1f}% reduction
↗ Val Loss: {val_improvement:.1f}% reduction

PERFORMANCE METRICS:
⏱ Total Time: {total_time:.1f}s ({total_time/60:.1f} min)
⏱ Avg Epoch: {avg_epoch_time:.1f}s
🎯 Data Sparsity: {active_pct:.1f}% active
🔧 Final Grad Norm: {final_grad_norm if isinstance(final_grad_norm, str) else f'{final_grad_norm:.3f}'}
📈 Final LR: {final_lr if isinstance(final_lr, str) else f'{final_lr:.2e}'}

DEEP MODEL FEATURES:
• Mixed Precision Training ✓
• Gradient Clipping ✓  
• Batch Normalization ✓
• Residual Connections ✓
• Dropout Regularization ✓
• Early Stopping ✓
    """
    
    axes[2, 1].text(0.05, 0.95, stats_text, transform=axes[2, 1].transAxes, 
                    fontsize=10, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 Deep training analysis saved to: {save_path}")
    plt.show()

def analyze_deep_model_convergence(history):
    """
    Analyze convergence characteristics specific to deep networks.
    """
    if not history or not history['train_loss']:
        print("❌ No training history available")
        return
    
    print("🔍 DEEP MODEL CONVERGENCE ANALYSIS")
    print("=" * 50)
    
    # Loss analysis
    train_losses = history['train_loss']
    val_losses = history['val_loss']
    
    print(f"\n📈 Loss Progression:")
    print(f"   Initial train loss: {train_losses[0]:.6f}")
    print(f"   Final train loss: {train_losses[-1]:.6f}")
    print(f"   Train improvement: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")
    
    if val_losses:
        print(f"   Initial val loss: {val_losses[0]:.6f}")
        print(f"   Final val loss: {val_losses[-1]:.6f}")
        print(f"   Best val loss: {min(val_losses):.6f}")
        print(f"   Val improvement: {((val_losses[0] - val_losses[-1]) / val_losses[0] * 100):.1f}%")
    
    # Convergence stability
    if len(train_losses) >= 10:
        last_10_std = np.std(train_losses[-10:])
        print(f"   Last 10 epochs std: {last_10_std:.6f} (stability indicator)")
    
    # Gradient norm analysis
    if history['gradient_norms']:
        grad_norms = history['gradient_norms']
        print(f"\n🎯 Gradient Analysis:")
        print(f"   Average grad norm: {np.mean(grad_norms):.3f}")
        print(f"   Max grad norm: {np.max(grad_norms):.3f}")
        print(f"   Final grad norm: {grad_norms[-1]:.3f}")
        print(f"   Clipping threshold: {config.gradient_clip_norm}")
        
        clipped_epochs = sum(1 for norm in grad_norms if norm >= config.gradient_clip_norm * 0.9)
        print(f"   Clipped epochs: {clipped_epochs}/{len(grad_norms)} ({100*clipped_epochs/len(grad_norms):.1f}%)")
    
    # Learning rate analysis
    if history['learning_rate']:
        lrs = history['learning_rate']
        print(f"\n📊 Learning Rate Analysis:")
        print(f"   Initial LR: {lrs[0]:.2e}")
        print(f"   Final LR: {lrs[-1]:.2e}")
        print(f"   Min LR: {min(lrs):.2e}")
        
        lr_reductions = sum(1 for i in range(1, len(lrs)) if lrs[i] < lrs[i-1] * 0.99)
        print(f"   LR reductions: {lr_reductions}")
    
    # Sparsity analysis
    if history['sparsity_stats']:
        final_stats = history['sparsity_stats'][-1]
        print(f"\n🎭 Dataset Sparsity:")
        print(f"   Active samples: {final_stats['active_samples']:,}")
        print(f"   Inactive samples: {final_stats['inactive_samples']:,}")
        print(f"   Active fraction: {100 * final_stats['active_fraction']:.1f}%")
    
    # Training efficiency
    if history['epoch_times']:
        times = history['epoch_times']
        print(f"\n⚡ Training Efficiency:")
        print(f"   Total time: {sum(times):.1f}s ({sum(times)/60:.1f} min)")
        print(f"   Average epoch: {np.mean(times):.1f}s")
        print(f"   Fastest epoch: {min(times):.1f}s")
        print(f"   Slowest epoch: {max(times):.1f}s")

# Generate comprehensive deep training analysis
print("🎨 Generating deep training analysis...")
plot_deep_training_analysis(training_history)
analyze_deep_model_convergence(training_history)